In [ ]:
# Retrieve the selected features, resampled training data, and encoded validation/test labels
%store -r selected_train_features_final

%store -r selected_test_features_final

%store -r resampled_train_features

%store -r resampled_train_labels

%store -r selected_valid_features

%store -r encoded_valid_labels

%store -r encoded_test_labels

In [ ]:
# Assign the resampled training labels for model training
selected_train_labels = resampled_train_labels

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV

# Initialize the XGBoost classifier for multiclass classification
xgb_model = XGBClassifier(
    objective="multi:softprob",
    num_class=7,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1
)

# Define a small hyperparameter search space
param_grid = {
    "n_estimators": [300, 500],
    "learning_rate": [0.05, 0.1],
    "max_depth": [6, 8],
    "min_child_weight": [1],
    "subsample": [0.8],
    "colsample_bytree": [0.8]
}

# Configure randomized cross-validation for hyperparameter tuning
random_search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_grid,
    n_iter=2,
    scoring="f1_macro",
    cv=2,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

# Train and evaluate the selected hyperparameter combinations
random_search.fit(
    selected_train_features_final,
    selected_train_labels
)

# Display the best hyperparameters
print("\nBEST PARAMETERS")
print("=" * 60)
print(random_search.best_params_)

# Display the best cross-validation macro F1 score
print("\nBEST CV MACRO F1:")
print(random_search.best_score_)

Fitting 2 folds for each of 2 candidates, totalling 4 fits

BEST PARAMETERS
{'subsample': 0.8, 'n_estimators': 500, 'min_child_weight': 1, 'max_depth': 6, 'learning_rate': 0.1, 'colsample_bytree': 0.8}

BEST CV MACRO F1:
0.9657435733240949


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score

# Retrieve the best XGBoost model from hyperparameter tuning
tuned_xgb_model = random_search.best_estimator_

# Generate predictions on the validation set
tuned_valid_predictions = tuned_xgb_model.predict(
    selected_valid_features
)

# Display the validation classification report
print("=" * 70)
print("TUNED XGBOOST - VALIDATION")
print("=" * 70)

print(
    classification_report(
        encoded_valid_labels,
        tuned_valid_predictions,
        digits=2
    )
)

# Calculate the validation macro F1 score
tuned_valid_macro_f1 = f1_score(
    encoded_valid_labels,
    tuned_valid_predictions,
    average="macro"
)

print("\nValidation Macro F1:", tuned_valid_macro_f1)

# Display the validation confusion matrix
print("\n" + "=" * 70)
print("CONFUSION MATRIX")
print("=" * 70)

print(
    confusion_matrix(
        encoded_valid_labels,
        tuned_valid_predictions
    )
)

TUNED XGBOOST - VALIDATION
              precision    recall  f1-score   support

           0       0.83      0.89      0.86       461
           1       0.93      0.95      0.94       324
           2       0.82      0.83      0.82       324
           3       0.93      0.95      0.94       324
           4       0.95      0.96      0.96     15460
           5       0.83      0.85      0.84       324
           6       0.87      0.84      0.86      4666

    accuracy                           0.93     21883
   macro avg       0.88      0.90      0.89     21883
weighted avg       0.93      0.93      0.93     21883


Validation Macro F1: 0.8885411972793553

CONFUSION MATRIX
[[  412     0     7     0    35     6     1]
 [    0   309     6     0     0     9     0]
 [    3    11   268     0     4    38     0]
 [    1     0     0   309     0     0    14]
 [   65     0    11     4 14828     4   548]
 [    0    12    34     0     4   274     0]
 [   15     0     2    18   697     0  3934]]


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score

# Retrieve the best XGBoost model from hyperparameter tuning
tuned_xgb_model = random_search.best_estimator_

# Generate predictions on the test set
tuned_test_predictions = tuned_xgb_model.predict(
    selected_test_features_final
)

# Display the test classification report
print("=" * 70)
print("TUNED XGBOOST - TEST")
print("=" * 70)

print(
    classification_report(
        encoded_test_labels,
        tuned_test_predictions,
        digits=2
    )
)

# Calculate the test macro F1 score
tuned_test_macro_f1 = f1_score(
    encoded_test_labels,
    tuned_test_predictions,
    average="macro"
)

print("\nTest Macro F1:", tuned_test_macro_f1)

# Display the test confusion matrix
print("\n" + "=" * 70)
print("CONFUSION MATRIX")
print("=" * 70)

print(
    confusion_matrix(
        encoded_test_labels,
        tuned_test_predictions
    )
)

TUNED XGBOOST - TEST
              precision    recall  f1-score   support

           0       0.85      0.92      0.88       460
           1       0.92      0.95      0.93       324
           2       0.82      0.77      0.79       324
           3       0.92      0.97      0.95       324
           4       0.95      0.96      0.96     15461
           5       0.76      0.86      0.81       324
           6       0.88      0.84      0.86      4667

    accuracy                           0.93     21884
   macro avg       0.87      0.90      0.88     21884
weighted avg       0.93      0.93      0.93     21884


Test Macro F1: 0.8829582928877725

CONFUSION MATRIX
[[  424     0     4     0    24     8     0]
 [    0   308     6     0     0    10     0]
 [    3    12   248     0     0    61     0]
 [    1     0     0   315     0     0     8]
 [   55     0    13     2 14874     8   509]
 [    2    15    29     0     0   278     0]
 [   14     0     4    25   703     0  3921]]
